<a href="https://colab.research.google.com/github/Jonnalasrikar/DSA/blob/main/AIResumeAnalyser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-community langchain-chroma langchain-huggingface langchain-ollama sentence-transformers langchain_openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 k

In [2]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

In [5]:
# How many characters in all the documents?

knowledge_base_path = "/content/drive/MyDrive/ai_resume_analyzer_dataset/data/**/*.txt"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 9 files in the knowledge base
Total characters in knowledge base: 1,460


In [6]:
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 333


In [7]:
folders = glob.glob("/content/drive/MyDrive/ai_resume_analyzer_dataset/data/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.txt", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)
print(f"Loaded {len(documents)} documents")

Loaded 9 documents


In [8]:
documents

[Document(metadata={'source': '/content/drive/MyDrive/ai_resume_analyzer_dataset/data/job_descriptions/java_backend_jd.txt', 'doc_type': 'job_descriptions'}, page_content='Job Role: Java Backend Developer\n\nRequired Skills:\n- Java\n- Spring Boot\n- REST APIs\n- MySQL\n- Docker\n\nResponsibilities:\n- Develop scalable backend services\n- Build REST APIs\n- Optimize database queries\n'),
 Document(metadata={'source': '/content/drive/MyDrive/ai_resume_analyzer_dataset/data/job_descriptions/ai_engineer_jd.txt', 'doc_type': 'job_descriptions'}, page_content='Job Role: AI Engineer\n\nRequired Skills:\n- Python\n- LangChain\n- Vector Databases\n- NLP\n- SQL\n\nResponsibilities:\n- Build RAG pipelines\n- Develop AI-powered applications\n'),
 Document(metadata={'source': '/content/drive/MyDrive/ai_resume_analyzer_dataset/data/skills/chromadb.txt', 'doc_type': 'skills'}, page_content='ChromaDB is a vector database used in RAG systems to store embeddings and perform semantic search.'),
 Documen

In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 9 chunks
First chunk:

page_content='Job Role: Java Backend Developer

Required Skills:
- Java
- Spring Boot
- REST APIs
- MySQL
- Docker

Responsibilities:
- Develop scalable backend services
- Build REST APIs
- Optimize database queries' metadata={'source': '/content/drive/MyDrive/ai_resume_analyzer_dataset/data/job_descriptions/java_backend_jd.txt', 'doc_type': 'job_descriptions'}


In [10]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectorstore created with 9 documents


In [11]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 9 vectors with 384 dimensions in the vector store


In [12]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])

vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']

doc_types = [metadata['doc_type'] for metadata in metadatas]

colors = [
    ['blue', 'green', 'red', 'orange', 'purple'][
        ['resumes', 'job_descriptions', 'skills', 'interview_questions', 'ats_rules'].index(t)
    ]
    for t in doc_types
]

In [13]:
from sklearn.manifold import TSNE

tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=5
)

reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=8, color=colors, opacity=0.8),
    text=[
        f"Type: {t}<br>Text: {d[:100]}..."
        for t, d in zip(doc_types, documents)
    ],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    width=800,
    height=600
)

fig.show()

In [14]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# 3D t-SNE
tsne = TSNE(
    n_components=3,
    random_state=42,
    perplexity=5
)

reduced_vectors = tsne.fit_transform(vectors)

# Create 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',

    marker=dict(
        size=6,
        color=colors,
        opacity=0.8
    ),

    text=[
        f"Type: {t}<br>Text: {d[:100]}..."
        for t, d in zip(doc_types, documents)
    ],

    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',

    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),

    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

In [15]:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [16]:
DB_NAME = "vector_db"

In [17]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
import torch

# Define the Hugging Face model to use for generation
HF_MODEL_NAME = "google/gemma-2b-it" # A popular free Hugging Face instruct model

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_NAME,
    torch_dtype=torch.bfloat16, # Use bfloat16 for efficiency if supported by GPU
    device_map="auto" # Automatically map model to available devices (GPU/CPU)
)

# Create a text generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512, # Max output tokens
    do_sample=True,
    temperature=0.7, # Adjust for creativity vs. consistency
    top_p=0.95, # Adjust for diversity
    num_return_sequences=1,
)

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'top_p', 'num_return_sequences', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [19]:
retriever = vectorstore.as_retriever()

In [20]:
llm = HuggingFacePipeline(pipeline=pipe)

In [21]:
SYSTEM_PROMPT_TEMPLATE = """
You are an expert AI Resume Analyzer and Job Matching Assistant.

Your tasks:
1. Analyze uploaded resumes
2. Identify candidate skills
3. Find missing skills
4. Match suitable jobs from database
5. Suggest improvements
6. Give ATS optimization tips

Use the provided context to answer.

Context:
{context}
"""

In [22]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 28.1 MB/s eta 0:00:00


In [23]:
# Install libraries for Hugging Face models
!pip install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.3 MB/s eta 0:00:00


In [24]:
from pypdf import PdfReader

In [25]:
def analyze_resume(file):

    try:

        from pypdf import PdfReader

        resume_text = ""

        # PDF Handling
        if file.endswith(".pdf"):

            reader = PdfReader(file)

            for page in reader.pages:

                text = page.extract_text()

                if text:
                    resume_text += text + "\n"

        # TXT Handling
        elif file.endswith(".txt"):

            with open(file, "r", encoding="utf-8") as f:
                resume_text = f.read()

        else:
            return "Unsupported file format."

        # Retrieve docs
        docs = retriever.invoke(resume_text)

        context = "\n\n".join(
            doc.page_content for doc in docs
        )

        system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
            context=context
        )

        user_prompt_content = f"""
        Analyze this resume carefully.

        Tasks:
        - Identify candidate skills
        - Find best matching jobs
        - Calculate ATS quality
        - Suggest improvements
        - Mention missing skills

        Resume:
        {resume_text}
        """
        # Combine system and user prompts into a single input for Gemma instruct model
        full_input_for_model = f"{system_prompt}\n\n{user_prompt_content}"

        # Apply Gemma's chat template for correct instruction following
        # For 'google/gemma-2b-it', a common template is:
        # <start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n
        formatted_prompt = f"<start_of_turn>user\n{full_input_for_model}<end_of_turn>\n<start_of_turn>model\n"

        # For HuggingFacePipeline, invoke returns the raw generated text (which includes the input prompt)
        raw_response_text = llm.invoke(formatted_prompt)

        # The model's actual response should be after the `formatted_prompt`.
        # This is a common pattern for text-generation pipelines, where the output
        # includes the input prompt. We need to strip the prompt part.
        if raw_response_text.startswith(formatted_prompt):
            model_generated_content = raw_response_text[len(formatted_prompt):].strip()
        else:
            # Fallback if the output format is unexpected
            model_generated_content = raw_response_text.strip()

        return model_generated_content

    except Exception as e:
        return str(e)

In [26]:
interface = gr.Interface(
    fn=analyze_resume,

    inputs=gr.File(
    label="Upload Resume",
    type="filepath"
),

    outputs=gr.Textbox(
        label="AI Resume Analysis",
        lines=25
    ),

    title="AI Resume Analyzer + Job Matching System",

    description="""
Upload a resume to:
- Analyze ATS quality
- Match jobs
- Identify skills
- Get improvement suggestions
"""
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c4a7ede8c8fdb89048.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
